<a href="https://colab.research.google.com/github/eeolga/deep/blob/main/Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An algorithm for pre-processing the activity record of an individual e-course log extract.

In [ ]:
import pandas as pd
import re

# Load the ’your_logs.xlsx’ to ensure you start with the original data
df = pd.read_excel('/content/your_logs.xlsx')

# Delete rows with 'tutor name' in the 'User full name' column
df = df[df['User full name'] != 'tutor name']

# Delete rows where 'Event context' starts with 'Course'
# and 'Deleted'

df = df[~((df['Event context'].astype(str).str.startswith('Course')))]
           (df['Event context'].astype(str).str.startswith('Deleted')))]

# Columns to delete (including 'User full name' now that filtering is done)

columns_to_delete = ['User full name', 'Affected user', 'Component', 'Event name', 'Origin', 'IP address']
df = df.drop(columns=columns_to_delete, errors='ignore')

# Extract the last number from 'Description' column for 'tool_code'
# This finds all numbers in the string and takes the last one; if no numbers, it assigns None
df['tool_code'] = df['Description'].astype(str).apply(lambda x: re.findall(r'\d+', x)[-1] if re.findall(r'\d+', x) else None)

# Delete the original 'Description' column as requested
df = df.drop(columns=['Description'], errors='ignore')

# Group by 'Event context' and 'tool_code' and count occurrences, naming the count column 'logs'
df_grouped = df.groupby(['Event context', 'tool_code']).size().reset_index(name='logs')

# Save the updated and grouped DataFrame to a new Excel file with specified columns
df_grouped[['Event context', 'tool_code', 'logs']].to_excel('preparation_1.xlsx', index=False)
print("Data processed, grouped, and saved to 'preparation_1.xlsx'")

df_preparation_1 = pd.read_excel('preparation_1.xlsx')

# First: Find the tool_code with the maximum logs for each 'Event context'
# Using idxmax to get the index of the row with the max 'logs' within each group
df_max_tool_code = df_preparation_1.loc[df_preparation_1.groupby('Event context')['logs'].idxmax()][['Event context', 'tool_code']].reset_index(drop=True)

# Second: Calculate the sum of 'logs' for each 'Event context'
df_sum_logs = df_preparation_1.groupby('Event con-text')['logs'].sum().reset_index()

# Third: Merge the two results to get the desired output
df_final_grouped = pd.merge(df_sum_logs, df_max_tool_code, on='Event context', how='left')

# Save the updated and grouped DataFrame to a new Excel file with specified columns
df_final_grouped[['Event context', 'tool_code', 'logs']].to_excel('preparation_2.xlsx', index=False)
print("Data processed and saved to 'preparation_2.xlsx'")


**Results:**


The preparation_2.xlsx file, which contains the ’Event Context,’ ’tool code,’ and the maximum number of ’logs’ for each context, is a valuable resource for inst-ructors to understand course dynamics and consider potential changes.
By reviewing these records, instructors can quickly determine which resour-ces or tools are most frequently used by students for each assignment or context. This allows them to highlight popular learning materials or interactive elements.
As subject-matter experts, instructors can use the identified tool codes (rep-resenting specific resources) from the preparation_2.xlsx file and rank them according to Bloom's Taxonomy levels (Remember/ Understand/ Apply/ Analyze/ Evaluate/ Create). By seeing which specific tools are used most frequently for specific assignments, instructors can:


i.	Analyse whether the most frequently used tools match the expected cog-nitive level for a given assignment.


ii.	Determine whether students gravitate towards lower-level tools when hi-gher-level tools are expected, or vice versa.


iii.	Make decisions about implementing or emphasising tools that support different levels of Bloom's Taxonomy to better structure learning or mo-tivate students accordingly.


The preparation_2.xlsx provides a clear overview of student engagement with specific resources, which can then be combined with the instructor's peda-gogical knowledge to make informed decisions about course design, resource allocation, and alignment with learning objectives and Bloom's Taxonomy.